In [ ]:
!pip install roboflow ultralytics

In [ ]:
from roboflow import Roboflow
#insertar la API de Roboflow
rf = Roboflow(api_key="")
#Insertar id del workspace, colocar id del project
project = rf.workspace("").project("")
#Insertar la versión del project, sustituir por el 2
version = project.version(2)
#insertar el modelo que se va a utilizar
#NOTA: En la fecha que se realizo este proyecto, Roboflow no soporta exportar las fuentes de datos para YOLO Segmentation por lo cual se usó coco-segmentation
dataset = version.download("coco-segmentation")


In [ ]:
#Este bloque de código permite estandarizar el nombre y extensión de los archivos que son soportados
import os
import json
import shutil

BASE_DIR = "/content/lineas2-2"
splits = ["train", "valid", "test"]
img_ext = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]


def normalize_filename(name):
    #Extrae solo el nombre del archivo y estandariza minúsculas.
    name = os.path.basename(name)
    return name.lower()


def map_images_in_folder(images_dir):
    #Crea un diccionario: nombre-normalizado -> nombre-real
    mapping = {}
    for f in os.listdir(images_dir):
        if any(f.lower().endswith(ext.lower()) for ext in img_ext):
            mapping[f.lower()] = f
    return mapping


def convert_coco_to_yolo_seg(coco_json_path, labels_dir, images_dir):
    #Convierte segmentación COCO a YOLO-seg.

    with open(coco_json_path, "r") as f:
        data = json.load(f)

    # Mapeo imagen-id-> filename normalizado
    id_to_filename = {}
    for img in data["images"]:
        fn = normalize_filename(img["file_name"])
        id_to_filename[img["id"]] = {
            "file": fn,
            "width": img["width"],
            "height": img["height"]
        }

    # Mapeo dentro de la carpeta real
    folder_map = map_images_in_folder(images_dir)

    for ann in data["annotations"]:
        img_id = ann["image_id"]

        if img_id not in id_to_filename:
            continue

        info = id_to_filename[img_id]
        W, H = info["width"], info["height"]
        file_norm = info["file"]

        # Buscar el archivo real puede tener distinta capitalización
        if file_norm not in folder_map:
            continue  # No encontró archivo -> no generamos label

        real_filename = folder_map[file_norm]

        segmentation = ann["segmentation"]
        if not segmentation:
            continue

        txt_name = os.path.splitext(real_filename)[0] + ".txt"
        txt_path = os.path.join(labels_dir, txt_name)

        class_id = ann["category_id"]

        with open(txt_path, "a") as f:
            for polygon in segmentation:
                if len(polygon) < 6:
                    continue

                normalized = []
                for i in range(0, len(polygon), 2):
                    x = polygon[i] / W
                    y = polygon[i+1] / H
                    normalized.extend([x, y])

                pts = " ".join([f"{p:.6f}" for p in normalized])
                f.write(f"{class_id} {pts}\n")


def organize_and_convert(split):
    split_path = os.path.join(BASE_DIR, split)
    print(f"Procesando {split_path}")

    if not os.path.exists(split_path):
        print("No existe, se omite.")
        return

    img_dir = os.path.join(split_path, "images")
    lbl_dir = os.path.join(split_path, "labels")

    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    # Mover imágenes
    for f in os.listdir(split_path):
        fpath = os.path.join(split_path, f)
        if os.path.isfile(fpath) and any(f.lower().endswith(ext.lower()) for ext in img_ext):
            shutil.move(fpath, os.path.join(img_dir, f))

    # Buscar JSON
    coco_json = None
    for f in os.listdir(split_path):
        if f.endswith(".json"):
            coco_json = os.path.join(split_path, f)
            break

    if coco_json is None:
        print("JSON no se encontro encontrado")
        return

    print("Convirtiendo COCO -> YOLO SEG…")
    convert_coco_to_yolo_seg(coco_json, lbl_dir, img_dir)

    shutil.copy(coco_json, os.path.join(lbl_dir, "annotations_coco.json"))

    print(f"{split} procesado correctamente")


# Ejecutar
for s in splits:
    organize_and_convert(s)

print("\nConversion FINAL completada")


In [ ]:
#Este bloque de código permite eliminar las imágenes que no coinciden con el nombre
import os

BASE = "/content/lineas2-2"

def remove_missing(split):
    print(f"\nLimpiando {split.upper()}")
    img_dir = os.path.join(BASE, split, "images")
    lbl_dir = os.path.join(BASE, split, "labels")

    count_removed = 0

    for img in os.listdir(img_dir):
        if not img.lower().endswith(('.jpg','.jpeg','.png')):
            continue

        txt_name = os.path.splitext(img)[0] + ".txt"
        txt_path = os.path.join(lbl_dir, txt_name)

        # Si no existe el .txt -> eliminar imagen
        if not os.path.exists(txt_path):
            os.remove(os.path.join(img_dir, img))
            print("Eliminada sin label:", img)
            count_removed += 1

    print(f"Total eliminadas en {split}: {count_removed}")

# Limpiar train y valid (test está bien)
for split in ["train", "valid"]:
    remove_missing(split)


In [ ]:
#Este bloque de código permite comprobar si todas las imágenes tienen etiqueta
import os

BASE = "/content/lineas2-2"

def check(split):
    img_dir = os.path.join(BASE, split, "images")
    lbl_dir = os.path.join(BASE, split, "labels")

    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    missing = []

    for img in imgs:
        txt = os.path.splitext(img)[0] + ".txt"
        if not os.path.exists(os.path.join(lbl_dir, txt)):
            missing.append(img)

    print(f"\n{split.upper()}: imágenes sin label = {len(missing)}")
    if missing:
        print(missing)

for s in ["train", "valid", "test"]:
    check(s)


In [ ]:
#Este bloque de código permite crear un archivo .yaml, en donde se colocan los nombres de los archivos, la categoría (train,va,test) y las clases (Line)
data_yaml_path = "/content/lineas2-2/data.yaml"

content = f"""
path: /content/lineas2-2

train: train/images
val: valid/images
test: test/images

names:
  0: Line
"""

with open(data_yaml_path, "w") as f:
    f.write(content.strip())

print("Archivo data.yaml creado en:", data_yaml_path)

# Mostrar contenido
with open(data_yaml_path, "r") as f:
    print("\nContenido de data.yaml:\n")
    print(f.read())



In [ ]:
#Este bloque de código permite comprobar si todas las imágenes tienen etiqueta
import os

BASE = "/content/lineas2-2"

def check_split(split):
    print(f"\nRevisando {split.upper()}")
    img_dir = os.path.join(BASE, split, "images")
    lbl_dir = os.path.join(BASE, split, "labels")

    imgs = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])

    missing = []
    empty = []

    for img in imgs:
        txt = os.path.splitext(img)[0] + ".txt"
        txt_path = os.path.join(lbl_dir, txt)

        if not os.path.exists(txt_path):
            missing.append(txt)
        else:
            if os.path.getsize(txt_path) == 0:
                empty.append(txt)

    print("Labels faltantes:", len(missing))
    for m in missing:
        print("   -", m)

    print("Labels vacíos:", len(empty))
    for e in empty:
        print("   -", e)


for split in ["train", "valid", "test"]:
    check_split(split)


In [ ]:
# limpia imágenes sin etiquetas
import os

BASE = "/content/lineas2-2"

def remove_missing(split):
    img_dir = os.path.join(BASE, split, "images")
    lbl_dir = os.path.join(BASE, split, "labels")

    for img in os.listdir(img_dir):
        if img.lower().endswith(('.jpg','.png','.jpeg')):
            txt = os.path.splitext(img)[0] + ".txt"
            if not os.path.exists(os.path.join(lbl_dir, txt)):
                os.remove(os.path.join(img_dir, img))

for split in ["train", "valid", "test"]:
    remove_missing(split)


In [ ]:
from ultralytics import YOLO
#colocar la version de YOLO
model = YOLO("yolo11m-seg.pt")

In [ ]:
data_path = "/content/lineas2-2/data.yaml"
results = model.train(data=data_path,
                          epochs=150,
                          patience=50,
                          imgsz=768,           # Clave 1: Reduce memoria
                          batch=4,             # Clave 2: Mantiene batch
                          amp=True,            # Clave 3: Precisión mixta
                          mosaic=0.1,          # Clave 4: Reduce carga
                          plots=True)          # Clave 5: permiter ver resultados

In [ ]:
#enpaqueta todo los resultados del entrenamiento
!zip -r /content/runs.zip /content/runs

In [ ]:
#descarga la carpeta
from google.colab import files
files.download('/content/runs.zip')